# Exercises XP: RAG with LangChain (Student)


## 0) Setup
Run the install cell once.


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface


In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS, DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={source_column: row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])


## 2) Split into chunks


In [ ]:
chunk_size = 500
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])


## 3) Vector store + retriever (FAISS)


In [ ]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    splits, embeddings, distance_strategy=DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")


## 4) Build the RAG chain


In [ ]:
llm_id = "google/flan-t5-small"
hf = pipeline(
    "text2text-generation", model=llm_id, max_new_tokens=200
)

llm = HuggingFacePipeline(pipeline=hf)

qa = RetrievalQA.from_chain_type(
    llm=llm, retriever=retriever, chain_type="stuff"
)

print("RAG chain ready")


## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

try:
    rag_result = qa.invoke({"query": q})
    rag_answer = rag_result["result"]
except Exception:
    try:
        rag_answer = qa.run(q)
    except Exception:
        rag_result = qa({"query": q})
        rag_answer = rag_result["result"]

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_answer)

sources = []
try:
    sources = rag_result.get("source_documents", [])
except NameError:
    pass
if sources:
    print("\nSources:")
    for d in sources:
        print("-", d.metadata.get("source"))
